In [1]:
# pip install requests pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
import pandas as pd
import time
import re
from tqdm import tqdm

HF_TOKEN = "hf_****"

API_URL = "https://router.huggingface.co/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json"
}


model_MetaLlama = {
    # "Meta-Llama-3-8B-Instruct": "meta-llama/Meta-Llama-3-8B-Instruct",
    # "Meta-Llama-3-70B-Instruct": "meta-llama/Meta-Llama-3-70B-Instruct",
    "Meta-Llama-3.2-1B-Instruct": "meta-llama/Llama-3.2-1B-Instruct",
}

model_Qwen = {
    "Qwen2.5-7B-Instruct": "Qwen/Qwen2.5-7B-Instruct",
    "Qwen2.5-Coder-7B-Instruct": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Qwen2.5-Coder-32B-Instruct": "Qwen/Qwen2.5-Coder-32B-Instruct",
}

model_DeepSeek = {
    "DeepSeek-R1-Distill-Qwen-7B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "DeepSeek-R1-Distill-Llama-8B": "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    "DeepSeek-R1-0528": "deepseek-ai/DeepSeek-R1-0528",
}

model_gemma = {
    "gemma-4-31B-it": "google/gemma-4-31B-it",
    "gemma-4-26B-A4B-it": "google/gemma-4-26B-A4B-it",
    "gemma-3-27b-it": "google/gemma-3-27b-it",
}

model_Kimi = {
    "Kimi-K2-Thinking": "moonshotai/Kimi-K2-Thinking",
    # "Kimi-K2-Instruct": "moonshotai/Kimi-K2-Instruct",
    # "Kimi-K2.5": "moonshotai/Kimi-K2.5",
}

# models = {"MetaLlama": model_MetaLlama, "Qwen": model_Qwen, "DeepSeek": model_DeepSeek, "gemma": model_gemma, "Kimi": model_Kimi}

models = {"MetaLlama": model_MetaLlama}


prompt_1 = """
I want to make three agents. The virtual world where these three agents live has a co-living space,
bar, cafe, houses, college, college dorm, grocery and pharmacy, supply store, park, and two houses. Can you create personas of all
three agents for me? I want you to provide me, with their Age, Educational Qualification, Personality Traits, Devices and technologies
they use, Work experience, Domain of work, Country, Gender with the following requirements:
Requirements:
• Names (mandatory): Ensure the names reflect a variety of ethnicities and faiths.
• Gender (mandatory): Include a balanced representation of different genders.
• Age (mandatory): Include a balanced representation of different genders.
• Personality Traits (mandatory): Include diverse personality traits
• Domain of Work (mandatory): Focus on diverse roles.
• Geographical Location (mandatory): Represent various regions globally.
• Few other mandatory requirement are education level, years of experience.
• Character Limit (optional): Each profile must be concise, within 300 characters.
"""

prompt_2 = """
Among these three agents, if you were to make one of them more vulnerable to phishing, who would you choose and why?
Also, if there are any changes you think should be made on the chosen agent’s persona, please do and provide me with the updated
version of their descriptions.
"""

prompts = {
    "prompt_1": prompt_1,
    "prompt_2": prompt_2
}



# =========================
# QUERY FUNCTION
# =========================
def query_model(model_name, prompt):
    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1200,
        "temperature": 0.7
    }

    try:
        r = requests.post(API_URL, headers=HEADERS, json=payload, timeout=60)
        # print("r", r)
        if r.status_code != 200:
            return ""
        # print(r.json())
        return r.json()["choices"][0]["message"]["content"]
    except:
        return ""


# =========================
# HELPERS
# =========================

def split_personas(text):
    """
    Heuristic split:
    - double newline OR numbered list OR 'Agent 1' style
    """
    # if not text:
    #     return []

    # blocks = re.split(r"\n\s*\n", text)

    # personas = []
    # for b in blocks:
    #     b = b.strip()

    #     # STRICT RULE: must be a real agent block
    #     if re.search(r"Agent\s*\d+\s*:", b):
    #         personas.append(b)

    # return personas

    # if not text:
    #     return []

    # blocks = re.split(r"\n\s*\n|(?=Agent\s*\d+)", text)
    # return [b.strip() for b in blocks if len(b.strip()) > 50]


    if not text:
        return []

    import re

    blocks = re.split(r"\n\s*\n", text)

    valid_personas = []

    for b in blocks:
        b_clean = b.strip()
        lower = b_clean.lower()

        # must contain at least ONE persona signal
        has_agent = False #re.search(r"agent\s*\d+", lower)
        has_name = "name" in lower
        has_age = "age" in lower

        if (has_name and has_age) and len(b_clean) > 50:
            valid_personas.append(b_clean)

    print(valid_personas)
    print(len(valid_personas))
    return valid_personas


def extract(pattern, text):
    match = re.search(pattern, text, re.IGNORECASE)
    return match.group(1).strip() if match else ""


def extract_persona_fields(persona_text):
    work_exp = extract(r"Work experience\s*[:\-]\s*(.*)", persona_text)

    # extract years from work experience
    years_exp = ""
    yoe = re.search(r"Years of experience\s*[:\-]\s*(.*)", persona_text)# Years of experience
    if yoe:
        years_exp = yoe.group(1)
    m = re.search(r"(\d+\+?\s*years?)", work_exp, re.IGNORECASE)
    if not years_exp and m:
        years_exp = m.group(1)

    name = ""
    m = re.search(r"Name\s*[:\-]\s*(.*)", persona_text)
    if m:
        name = m.group(1).strip()

    # fallback if format differs
    if not name:
        m = re.search(r"Agent\s*\d+\s*:\s*(.*)", persona_text)
        if m:
            name = m.group(1).strip()

    return {
        "Name": name,
        "Age": extract(r"Age\s*[:\-]\s*(\d+)", persona_text),
        "Gender": extract(r"Gender\s*[:\-]\s*(.*)", persona_text),
        "Personality Traits": extract(r"Personality Traits\s*[:\-]\s*(.*)", persona_text),
        "Domain of work": extract(r"Domain of Work\s*[:\-]\s*(.*)", persona_text),

        # "Work Experience": work_exp,
        "Years of Exp.": years_exp,

        "Location": extract(r"Country\s*[:\-]\s*(.*)", persona_text),
        "Education Level": extract(r"Educational Qualification\s*[:\-]\s*(.*)", persona_text),
        "Devices and Technologies Used": extract(r"Devices.*?technologies.*?[:\-]\s*(.*)", persona_text),
    }


def detect_phishing(persona_name, phishing_text):
    """
    Robust fuzzy detection:
    - checks name presence
    - checks P1/P2/P3 mapping
    """
    if not phishing_text:
        return "N", ""

    lower = phishing_text.lower()
    name_hit = persona_name.lower() in lower

    if name_hit:
        return "Y", phishing_text

    # fallback: keyword vulnerability signals
    vuln_keywords = ["most vulnerable", "easily tricked", "high risk", "susceptible"]
    if any(k in lower for k in vuln_keywords):
        return "Y", phishing_text

    return "N", ""


def get_vulnerable_persona(phishing_text):

    if not phishing_text:
        return None, ""

    text = phishing_text.lower()

    # Agent X
    m = re.search(r"agent\s*(\d)", text)
    if m:
        return f"P{m.group(1)}", phishing_text

    # Persona X fallback
    m = re.search(r"persona\s*(\d)", text)
    if m:
        return f"P{m.group(1)}", phishing_text

    return None, phishing_text

# =========================
# MAIN PIPELINE
# =========================
rows = []



for group_name, model_group in models.items():
    print(group_name)
    
    for model_name, model_id in model_group.items():
        count = 1
        
        print(f"\nRunning: {model_name}")

        for run in tqdm(range(100)):
            print(count)
            if count==0:
                break

            # STEP 1: personas
            persona_output = query_model(model_id, prompt_1)
            # print(persona_output)
            personas = split_personas(persona_output)
            print(personas)
            if len(personas) == 0:
                continue
                
            # STEP 2: phishing analysis
            chained_prompt = f"""
Here are three agents:

{persona_output}

{prompt_2}
"""
            phishing_output = query_model(model_id, chained_prompt)

            print(phishing_output)
            time.sleep(2)

            # DETECT VULNERABLE PERSONA (ONLY ONE)
            vulnerable_pid, reason_text = get_vulnerable_persona(phishing_output)

            # STEP 3: build structured dataset
            flag = False
            for i, persona in enumerate(personas):

                print('----', persona)
                
                fields = extract_persona_fields(persona)
                persona_name = fields["Name"] if fields["Name"] else f"Persona_{i+1}"

                label, reason = detect_phishing(persona_name, phishing_output)

                persona_id = f"P{i+1}"
                if vulnerable_pid == persona_id:
                    label = "Y"
                    reason = reason_text
                else:
                    label = "N"
                    reason = ""


                rows.append({
                    "Model": model_name,
                    "Persona ID": f"P{i+1}",
                    "Persona Name": persona_name,
                    "Profile details": persona,
                    **fields,
                    "Phishing Susceptible (Y/N)": label,
                    "Reasons": reason
                })
                flag = True
            if flag == True:
                count=count-1
                



    # =========================
    # SAVE CSV
    # =========================
    df = pd.DataFrame(rows)
    df.to_csv(f"hf_router_dataset_11_{group_name}.csv", index=False)

    print(f"\nDataset saved as hf_router_dataset_{group_name}.csv")

MetaLlama

Running: Meta-Llama-3.2-1B-Instruct


  0%|                                                   | 0/100 [00:00<?, ?it/s]

1
['Here are three personas of agents, each with a name, gender, age, personality traits, devices and technologies, work experience, domain of work, geographical location, and education level.', "- **Name:** Amira Ali (Female, 32)\n- **Age:** 32 (Female: 28, Male: 4)\n- **Personality Traits:** Curious, Resourceful, Empathetic\n- **Devices and Technologies:** Smartphone, Laptop, Smartwatch\n- **Work Experience:** 5 years of experience in Customer Service and Human Resources\n- **Domain of Work:** Human Resources and Customer Service\n- **Country:** United States of America\n- **Education Level:** Bachelor's Degree in Business Administration", "- **Name:** Khaled Khan (Male, 35)\n- **Age:** 35 (Male: 28, Female: 7)\n- **Personality Traits:** Confident, Outgoing, Analytical\n- **Devices and Technologies:** Smartphone, Laptop, Tablet\n- **Work Experience:** 10 years of experience in Marketing and Sales\n- **Domain of Work:** Marketing and Sales\n- **Country:** United Kingdom\n- **Education

  1%|▍                                          | 1/100 [00:07<12:36,  7.65s/it]

---- Here are three personas of agents, each with a name, gender, age, personality traits, devices and technologies, work experience, domain of work, geographical location, and education level.
---- - **Name:** Amira Ali (Female, 32)
- **Age:** 32 (Female: 28, Male: 4)
- **Personality Traits:** Curious, Resourceful, Empathetic
- **Devices and Technologies:** Smartphone, Laptop, Smartwatch
- **Work Experience:** 5 years of experience in Customer Service and Human Resources
- **Domain of Work:** Human Resources and Customer Service
- **Country:** United States of America
- **Education Level:** Bachelor's Degree in Business Administration
---- - **Name:** Khaled Khan (Male, 35)
- **Age:** 35 (Male: 28, Female: 7)
- **Personality Traits:** Confident, Outgoing, Analytical
- **Devices and Technologies:** Smartphone, Laptop, Tablet
- **Work Experience:** 10 years of experience in Marketing and Sales
- **Domain of Work:** Marketing and Sales
- **Country:** United Kingdom
- **Education Level:**

In [2]:
Meta-Llama-3-8B-Instruct -> 10
Meta-Llama-3-70B-Instruct -> 10
Meta-Llama-3.2-1B-Instruct -> 4

SyntaxError: invalid decimal literal (618861861.py, line 1)